In [2]:
source = "/lium/buster1/larcher/M2/Projects/2026/xvectors/"
import numpy as np

In [3]:
Dic_idx = np.load(source+"voxblink2_idx.npy")
Dic = np.load(source+"voxblink2.npy")

In [ ]:
# Step 1 : Récuperer le residu
# r = X
# Step 2 : Comparer le résidus aux prototypes
# comp = r @ V
# Step 3 : Recuperer le argmax des scores
# max = argmax(abs(comp))
# Step 4 : Soustraire au résidu ce qu'on explique avec le prototype choisi
# r = r - V
# Step 5 : iterer K fois

In [4]:
def OMP(X, Dic, K):
    r = X.copy()
    D = Dic.shape[1]
    z = np.zeros(D)
    
    for _ in range(K):
        # 1 - Comparer le résidus aux prototypes
        scores = Dic.T @ r

        # 2 - Selectionner celui qui explique le mieux notre résidu
        j = np.argmax(np.abs(scores))
        c = scores[j]

        
        # 3 - Soustraire du résidu ce qui est expliquer par le prototype et l'ajouter à l'interpretation
        z[j] += c
        r = r - c * Dic[:, j]
    
    return z, r  

In [16]:
d, D, K = 10, 30, 5
Ddic = np.random.randn(d, D)
Ddic /= np.linalg.norm(Ddic, axis=0)   # normalisation des colonnes

x = np.random.randn(d)
z, r = OMP(x, Ddic, K)

# Vérification : x ≈ reconstruction + résidu
x_hat = Ddic @ z
print("x - x_hat - r : ",np.linalg.norm(x - x_hat - r))   # doit être ~0 (identité algébrique)
print("r : ", np.linalg.norm(r))               # doit décroître si tu augmentes K

x - x_hat - r :  2.8066088919804617e-16
r :  0.7899122941591331


In [1]:
import torch
import torch.nn as nn


class CayleyRotations(nn.Module):
    """K rotations R_k ∈ SO(d), paramétrées par la transformée de Cayley."""

    def __init__(self, d, K, init_scale=1e-3):
        super().__init__()
        self.d, self.K = d, K
        # Paramètres libres : on en déduit A antisymétrique dans skew()
        self.W = nn.Parameter(init_scale * torch.randn(K, d, d))

    def skew(self):
        # A = W - W^T  =>  A^T = -A  (forme (K, d, d))
        return self.W - self.W.transpose(-1, -2)

    def forward(self):
        """Retourne toutes les rotations, forme (K, d, d)."""
        A = self.skew()
        I = torch.eye(self.d, device=A.device, dtype=A.dtype).expand_as(A)
        # R = (I - A)(I + A)^{-1} = (I + A)^{-1}(I - A)  (les deux commutent)
        return torch.linalg.solve(I + A, I - A)

    def get_R(self, k):
        return self.forward()[k]

In [5]:
torch.manual_seed(0)
d, K = 8, 3
rot = CayleyRotations(d, K, init_scale=0.5).double()
R = rot()

I = torch.eye(d, dtype=R.dtype)
print("orthogonalité :", (R.transpose(-1, -2) @ R - I).norm(dim=(-1, -2)))
print("déterminants  :", torch.linalg.det(R))

x = torch.randn(10, d, dtype=R.dtype)
print("normes conservées :", torch.allclose(x.norm(dim=1), (x @ R[0].T).norm(dim=1)))

# Gradient
rot = CayleyRotations(d, K)
rot()[0].sum().backward()
print("gradient reçu :", rot.W.grad is not None and rot.W.grad.abs().sum() > 0)

# Init : A ≈ 0 => R ≈ I
print("init proche de l'identité :", (rot()[0] - torch.eye(d)).norm())

orthogonalité : tensor([8.5715e-16, 1.0035e-15, 1.0905e-15], dtype=torch.float64,
       grad_fn=<LinalgVectorNormBackward0>)
déterminants  : tensor([1.0000, 1.0000, 1.0000], dtype=torch.float64,
       grad_fn=<LinalgDetBackward0>)
normes conservées : True
gradient reçu : tensor(True)
init proche de l'identité : tensor(0.0232, grad_fn=<LinalgVectorNormBackward0>)
